# Equity standard factor-model benchmarks

Compare CAPM, FF3, Carhart4, FF5, FF6 and AQR BAB/QMJ extensions across 130 test portfolios. Includes time-series fit, rolling exposure attribution, static and Fama–MacBeth prices, identification diagnostics, spanning and snapshot export. The attribution scores use realized contemporaneous factors with previously estimated betas; they are not return forecasts.

Uses the checked-out ERP 0.13.0 source. For residual and dynamic pricing research use `01_equity_factor_discovery_and_pricing.ipynb`. Historical evidence is in `results/evidence/`.

In [ ]:
# Load checked-out ERP source. Install the repository first; see README.
import sys, subprocess, importlib.util
from pathlib import Path
from IPython.display import display
import numpy as np
import pandas as pd
roots = [Path.cwd(), *Path.cwd().parents, Path('/content/empirical-risk-premia-engine')]
REPO = next((p for p in roots if (p/'src'/'erp'/'__init__.py').is_file()), None)
if REPO is None:
    raise RuntimeError('Open from the cloned repository or extract the release to /content/empirical-risk-premia-engine; see README.')
sys.path.insert(0, str(REPO/'src'))
for key in list(sys.modules):
    if key == 'erp' or key.startswith('erp.'):
        del sys.modules[key]
from erp import (__version__,EquityData,EquityPremiaEngine,analyze_equity_residuals,
                 export_snapshot,RatesData,RatesPremiaEngine,RatesEmpiricalBondPremiaEngine,
                 load_macro_panel,load_macro_series_files,analyze_rates_v09)
print('Loaded ERP',__version__)

assert __version__ == "0.13.0"


## Configuration

Select a standard/augmented model and research universe. Set `USE_AQR = False` for a French-only offline-vendor compatibility run (`MODEL="FF6"`). If AQR disallows auto-download, upload the *original monthly* `.xlsx` files to your notebook directory and specify local paths. No long-only ETF substitution.

In [ ]:
MODEL = "FF6_BAB_QMJ"    # CAPM FF3 CARHART4 FF5 FF6 FF6_BAB FF6_QMJ FF6_BAB_QMJ FF3_BAB_QMJ
UNIVERSE = "POOLED_130"   # SIZE_BM_25 INDUSTRY_30 SIZE_OP_25 SIZE_INV_25 SIZE_MOM_25 POOLED_130
AS_OF = "2026-10-31"      # strict ISO date cutoff; change when new data become available
BETA_WINDOW = 60
USE_AQR = True
AQR_PATHS = {}             # e.g. {"BAB":"/content/BAB.xlsx", "QMJ":"/content/QMJ.xlsx"}
CACHE_DIR = Path("erp_data_cache")
OUT = Path("erp_v09_equity_output")
print("Model:", MODEL, "| Universe:",UNIVERSE, "| as_of:",AS_OF)

## 1 — Official public factors and test portfolios

Downloads 25 Size×B/M, 30 Industries, 25 Size×Profitability, 25 Size×Investment and 25 Size×Momentum, plus published BAB and QMJ **monthly long/short** factors. Uses full sample intersection and preserves FF3-family vs FF5-family SMB conventions.

In [ ]:
data = EquityData.from_public_sources(
    cache_dir=CACHE_DIR, aqr=USE_AQR, aqr_paths=AQR_PATHS,
)
print("Asset universes:",[(x, y.shape) for x,y in data.universes.items()])
print("AQR factors:", None if data.aqr is None else data.aqr.dropna().shape)
engine = EquityPremiaEngine(data)
f,rf = data.factors(MODEL)
display(f.loc[:AS_OF].tail())

## 2 — Model horse race: identical dates

The horse race computes cross-model time-series alpha/errors on **exactly the same calendar months**. GRS and FMB diagnostics are reported separately; don't confuse lower in-sample alpha RMSE with stronger out-of-sample return forecasts.

In [ ]:
MODELS = ["CAPM", "FF3", "CARHART4", "FF5", "FF6"]
if USE_AQR:
    MODELS += ["FF6_BAB", "FF6_QMJ", "FF6_BAB_QMJ"]
common_models = engine.compare_models(MODELS, UNIVERSE, as_of=AS_OF, beta_window=BETA_WINDOW)
display(common_models.sort_values("alpha_rmse_ann"))

## 3 — Fit selected factor specification, extract risks, errors and premia

Ex post factor attribution: `factor_component` is calculated from **realized current factor return**, using only beta estimated from prior months. It is not an ex-ante expected return forecast.

In [ ]:
snapshot = engine.fit(MODEL, UNIVERSE, as_of=AS_OF, beta_window=BETA_WINDOW, run_fmb=True)
print("Effective data through", snapshot.as_of, "| months",snapshot.metadata['n_months'])
print("GRS (standard, if identifiable):",snapshot.metadata['grs'])
display(snapshot.fit_diagnostics.sort_values("alpha_ann").head(12))
display(snapshot.betas.head())
display(snapshot.factor_premia.to_frame("mean_excess_monthly"))
print("Static cross-sectional prices of risk (monthly) and pricing errors:")
display(snapshot.static_cs_premia.to_frame("static_CS_lambda_monthly"))
display(snapshot.static_pricing_errors.to_frame("static_CS_error_monthly").head(8))
print("Static CS diagnostics:", snapshot.metadata["static_cross_sectional_pricing"])

## 4 — Fama–MacBeth: prices of risk + identification quality

Lagged betas use `t-60,...,t-1`. At each month t, contemporaneous cross-sectional returns determine a realized λ_t. The table juxtaposes λ estimates (and HAC t) with cross-sectional beta dispersion, partial dispersion, auxiliary R² and scaled condition number. This is **not** Shanken-adjusted inference.

In [ ]:
display(snapshot.fmb_diagnostics.round(5))
print("Historical traded-factor means vs FMB lambda estimates (monthly decimals):")
a = snapshot.factor_premia.rename("traded_factor_mean")
b = snapshot.fmb_diagnostics["lambda_monthly"].rename("FMB_lambda")
display(pd.concat([a,b],axis=1).round(5))
print("Last 5 monthly lambda observations:")
display(snapshot.fmb_lambdas.tail().round(4))
if USE_AQR:
    print("Official-factor spanning vs FF6 (not a time-series trading signal):")
    display(pd.DataFrame([engine.spanning_test(target="BAB", against_model="FF6", as_of=AS_OF),
                          engine.spanning_test(target="QMJ", against_model="FF6", as_of=AS_OF)]))

## 5 — Risk exposure / alpha / residual decomposition

`alpha` is the in-sample regression intercept. `residual_returns` are idiosyncratic regression residuals. They do **not** establish a monetizable future alpha.

In [ ]:
print("Alpha (annualized, first 10 assets)")
display(snapshot.alpha.head(10).mul(12).to_frame("fitted_alpha_ann"))
print("Realized factor-attribution sample (beta known strictly before each observation)")
display(snapshot.oos_attribution.tail(12))
print("Fitted residual covariance (first 5 assets)")
display(snapshot.residuals.iloc[:,-5:].cov().round(5))

## 6 — Export to common file schema

Exports CSV tables plus `manifest.json` and explicit meanings/limitations, consumable by Slicer/Referee without modifying their own statistical logic.

In [ ]:
destination = export_snapshot(snapshot, OUT / (UNIVERSE + "__" + MODEL))
print("Exported:",destination)
print("Files:",sorted(x.name for x in destination.iterdir()))
from erp.integrations import to_alpha_referee_factor_library
print("Alpha Referee factor library:", list(to_alpha_referee_factor_library(snapshot)["factors"].columns))
# Alpha Slicer v0.5.1 integration, only if you have actual strategy & benchmark time series:
# from erp.integrations import to_alpha_slicer_inputs
# slicer_kwargs = to_alpha_slicer_inputs(snapshot, strategy_net=real_net,
#                                       strategy_gross=real_gross, benchmark=real_benchmark)
# slicer_result = your_existing_slicer.fit(**slicer_kwargs)

## 7 — Research interpretation

Interpret each result separately: **factor fit**, **cross-sectional price of risk**, **identification strength**, **spanning**, **OOS attribution** and **predictive usefulness** are different questions. Re-run the same dates for comparisons with published QMJ/BAB; do not infer a universal best model. The historical v0.1 notebook remains unchanged and is included in the source ZIP as provenance.